In [0]:
dbutils.widgets.text('p_batch_id', '')

In [0]:
v_batch_id = dbutils.widgets.get('p_batch_id')
v_batch_id

In [0]:
%run ../00-common/01.environment-config


In [0]:
%run ../00-common/02.bronze-helpers

## Step 1 - Reading circuits file

In [0]:
source_file = f'{landing_folder_path}/{v_batch_id}/circuits.csv'

In [0]:
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

circuits_schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType()),
])

In [0]:
circuits_df = (
    spark.read
        .format('csv')
        .option('header', True)
        # .option('inferSchema', True) ## good for development but not for production becaused the data can change
        .schema(circuits_schema)
        .load(source_file)
)   

## Step 2 - Adding metadata

- Ingestion timestamp
- Filename

In [0]:
circuits_df_final = add_ingestion_metadata(circuits_df)

In [0]:
display(circuits_df_final)

## Step 3 - Writing to bronze delta table

In [0]:
write_to_bronze (
    final_df = circuits_df_final,
    target_table = f'{catalog_name}.{bronze_schema}.circuits',
    batch_id = v_batch_id
)

In [0]:
# circuits_df_final = circuits_df_final.withColumn('batch_id', F.lit(v_batch_id))

In [0]:
# (
#     circuits_df_final.write
#         .format('delta')
#         .mode('overwrite')
#         .partitionBy('batch_id')
#         .option('replaceWhere', f'batch_id = {v_batch_id}') # replace only the batch id partition
#         .saveAsTable(f'{catalog_name}.{bronze_schema}.circuits')
# )

In [0]:
%sql
SELECT * FROM formula1.bronze.circuits